# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead |Jannik Urban |  |
| Member | Mohammad Aman Ahmad |  |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** *Identifying and Predicting Successful Prediction Markets on Polymarket*

**Research question:**  
*Can machine learning identify distinct types of prediction markets on Polymarket, and do these market characteristics help predict market success measured by trading volume? Additionally, we investigate whether market rewards causally affect market success.*

**Why it matters:**  
*Prediction markets are increasingly used to forecast events in politics, economics, and sociology. Trading volume is especially relevant because it reflects market participation, trader interest, and overall market activity. In addition, Polymarket uses reward programs to incentivize liquidity providers and improve market quality. Understanding whether these rewards causally increase trading volume, and whether market characteristics identified through machine learning improve volume prediction, can provide valuable insights for market design, incentive mechanisms, and the efficiency of prediction markets.*


## 3. Data

**Source(s):**  
*Public available data from Polymarkets (licence/terms of use: https://docs.polymarket.com/api-reference/authentication) 
url: https://gamma-api.polymarket.com/markets/keyset *

**Unit of observation:** *One row represents one prediction market listed on Polymarket.*

**Key variables:**

| Variable              | Type               | Role       | Description                                                  |
| --------------------- | ------------------ | ---------- | ------------------------------------------------------------ |
| `id`                  | numeric | identifier | Unique identifier of the prediction market                   |
| `question`            | text               | feature    | Market question describing the predicted event               |
| `startDateIso`        | datetime           | feature    | Planned market start date                                    |
| `endDateIso`          | datetime           | feature    | Planned market end date                           |
| `volume`              | numeric            | target    | Total trading volume within the market                       |
| `liquidity`           | numeric            | feature     | Market liquidity                  |
| `outcomePrices`       | numeric/list       | feature    | Current implied probabilities of market outcomes             |
| `rewardsMinSize`      | numeric            | feature    | Minimum trade size eligible for rewards/incentives           |
| `restricted`          | boolean            | feature    | Indicates whether the market has access restrictions         |
| `spread`              | numeric            | feature    | Bid-ask spread indicating market efficiency                  |
| `bestBid`             | numeric            | feature    | Highest outstanding buy order price                          |
| `bestAsk`             | numeric            | feature    | Lowest outstanding sell order price                          |
| `active`              | boolean            | feature    | Indicates whether the market is currently active             |
| `closed`              | boolean            | feature    | Indicates whether the market has already closed              |
| `category`            | categorical        | feature    | Topic category of the market (e.g. politics, sports, crypto) |


**Potential data quality issues:**  
*Missing values in category, liquidity and volume. A few wrong dates in endDateIso.*


In [ ]:
# Data loading & first inspection
# ── replace with your own code ──────────────────────────────────────────
# for 200.000 entries

import asyncio
import aiohttp
import pandas as pd
from tqdm.notebook import tqdm

BASE_URL = "https://gamma-api.polymarket.com/markets/keyset"

TARGET_ROWS = 200_000
LIMIT = 500

all_markets = []


async def fetch_page(session, cursor=None):

    params = {
        "limit": LIMIT,
        "closed": "true"
    }

    # nur hinzufügen wenn vorhanden
    if cursor:
        params["after_cursor"] = cursor

    for attempt in range(5):

        try:

            async with session.get(BASE_URL, params=params) as response:

                if response.status != 200:
                    text = await response.text()
                    print("STATUS:", response.status)
                    print(text)

                response.raise_for_status()

                return await response.json()

        except Exception as e:

            print(f"Fehler: {e} | Versuch {attempt+1}/5")

            await asyncio.sleep(2)

    return None


async def main():

    global all_markets

    timeout = aiohttp.ClientTimeout(total=120)

    connector = aiohttp.TCPConnector(limit=10)

    async with aiohttp.ClientSession(
        timeout=timeout,
        connector=connector
    ) as session:

        cursor = None

        with tqdm(total=TARGET_ROWS) as pbar:

            while len(all_markets) < TARGET_ROWS:

                data = await fetch_page(session, cursor)

                if data is None:
                    print("Abbruch wegen mehrfacher Fehler")
                    break

                markets = data.get("markets", [])

                if len(markets) == 0:
                    print("Keine weiteren Markets")
                    break

                all_markets.extend(markets)

                pbar.update(len(markets))

                cursor = data.get("next_cursor")

                # letztes page token
                if not cursor or cursor == "LTE=":
                    print("Ende erreicht")
                    break

    return pd.DataFrame(all_markets[:TARGET_ROWS])


df_markets = await main()

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)
- [x] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification: We use a causal graph (DAG) to model the assumed relationships between market characteristics, market uncertainty and trading volume. In particular, we investigate whether market uncertainty, measured through implied outcome probabilities, causally affects market success measured by trading volume. Backdoor adjustment will be used to control for confounding variables identified in the DAG and to better isolate the causal effect of uncertainty on market success.*

### 4b. Supervised Learning
- [x] Linear / Ridge / Lasso regression
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] Decision Tree / Random Forest
- [ ] Neural network (regression or classification)
- [ ] Other: ___

*Justification: Random Forests are well suited for financial and market data because they can capture nonlinear relationships and interactions between market characteristics. The method is robust to noisy data and allows us to analyze feature importance when predicting market liquidity.

Linear and regularized regression models provide an interpretable benchmark for predicting market success. Ridge and Lasso regression additionally help address multicollinearity and reduce overfitting in the presence of correlated market features.*

### 4c. Unsupervised Learning / Generative Models
- [x] K-Means clustering
- [x] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification: K-Means clustering will be used to identify distinct groups of prediction markets based on their characteristics such as liquidity, spread, volume, and category. This helps uncover structural market types and supports exploratory analysis of market behavior.

Hierarchical clustering complements K-Means by allowing us to explore the nested structure and similarity between prediction markets. The method helps validate cluster assignments and provides additional insights into different market types.*


## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

- The metric(s) you will use for each model (e.g. RMSE, accuracy, AUC, silhouette score).

We will evaluate the performance of our supervised learning models using regression metrics such as RMSE (Root Mean Squared Error) and R² to measure how accurately the models predict market success measured by trading volume.

For clustering methods, we will use both the silhouette score and the elbow method to assess the quality and separation of the identified market clusters better and compare the results of both methods.

- How you will validate causal claims (e.g. refutation tests, sensitivity analysis).

We will conduct sensitivity analyses to evaluate whether the observed relationships remain stable under different model specifications and feature selections.

- Any baselines or benchmarks you will compare against.

As benchmarks, we will compare the performance of the Random Forest model against simpler baseline models such as Linear/Ridge/Lasso Regression and a naive baseline using the average trading volume prediction. For clustering, we will compare the results of K-Means and Hierarchical Clustering to assess the robustness of the identified market groups.


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 |Jannik| Data collection & cleaning     |
| 2 | both| EDA |
| 3 | Aman | Causal inference block |
| 4 | both | Supervised learning block |
| 5 | Jannik | Unsupervised / generative block |
| 6 | both | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*# Synthesis & Communication

This project combines **unsupervised learning, causal inference, and supervised learning** to better understand the characteristics and success of prediction markets on Polymarket.

First, we applied **unsupervised learning** to identify distinct market types based on structural market characteristics such as duration, reward size, categories, and outcome format. K-Means and Hierarchical Clustering revealed meaningful market segments, demonstrating that prediction markets are not homogeneous but can be grouped into different market profiles.

Next, we used **causal inference** to move beyond correlation and investigate whether reward mechanisms actually increase trading volume. By constructing a Directed Acyclic Graph (DAG), identifying the appropriate adjustment set, and estimating the Average Treatment Effect (ATE), we found evidence that markets with rewards achieve significantly higher trading volume. Refutation tests further supported the robustness of this causal relationship.

Finally, we developed **supervised learning models** to predict market success measured by trading volume. Linear Regression, Ridge, Lasso, and Random Forest models were evaluated, with the Random Forest using a log-transformed target variable achieving the best predictive performance. The cluster labels generated in the unsupervised learning stage were incorporated as additional features, allowing us to assess whether market segmentation improves prediction. While the different clustering approaches produced very similar predictive performance, they provided additional insights into market structure.

Overall, the three analytical approaches complement each other. **Unsupervised learning** identifies hidden market structures, **causal inference** estimates the effect of reward mechanisms on trading activity, and **supervised learning** leverages these insights to predict market success. Together, these methods provide a comprehensive understanding of prediction market behavior and offer valuable implications for market design, liquidity incentives, and platform optimization.*


# Limitations

While this study provides valuable insights into prediction market behavior, several limitations should be acknowledged.

First, the analysis relies solely on data from Polymarket, limiting the generalizability of the findings to other prediction market platforms. Second, some potentially relevant variables, such as bid-ask spread and order book information, were intentionally excluded from the supervised learning models because they are only observed after trading has begun and would introduce data leakage. Third, although the causal inference analysis controls for observed confounders, unobserved factors may still influence both reward allocation and trading volume. Finally, the clustering methods identified meaningful market segments, but different clustering algorithms produced very similar predictive performance, suggesting that additional market features may be needed to further improve prediction accuracy.

# Research Question

Our findings suggest that machine learning can successfully identify distinct types of prediction markets on Polymarket through unsupervised clustering techniques. These market segments capture meaningful differences in market characteristics and provide additional information for prediction models.

Furthermore, the supervised learning models demonstrate that trading volume can be predicted with reasonable accuracy, with Random Forest using a log-transformed target variable achieving the best performance. While the inclusion of cluster information only marginally improved prediction performance, it enhanced the interpretation of different market types.

Finally, the causal inference analysis provides evidence that market rewards have a positive causal effect on trading volume. This suggests that Polymarket's reward mechanism successfully incentivizes market activity and contributes to higher market participation.

Overall, the results indicate that combining unsupervised learning, causal inference, and supervised learning offers a comprehensive framework for understanding and predicting the success of prediction markets.